## Max Rectangle - Matrix

Matrix - list of rows in python
    

min_h = float('inf') = start with "infinity" so any number is smaller

for k... = check each column in the range

if heights[k] < min_h = keep the smallest

width = ... = count columns

area = ... = multiply

if area > best = keep the best

┌─────────────────────────────────────┐ │ ROW LOOP (i) │ │ "Slide the floor down" │ │ │ │ ┌─────────────────────────────┐ │ │ │ COLUMN LOOP (j) │ │ │ │ "Update the bar" │ │ │ │ heights[j] grows or breaks │ │ │ └─────────────────────────────┘ │ │ │ │ [Heights now looks like: 2, 1, 3] │ │ │ │ ┌─────────────────────────────┐ │ │ │ RANGE LOOP (left, right) │ │ │ │ "Check this rectangle" │ │ │ │ - Find shortest bar │ │ │ │ - Count bars (width) │ │ │ │ - Area = short × count │ │ │ └─────────────────────────────┘ │ └─────────────────────────────────────┘

[Row Loop i] 
    ↓ 
updates [Heights Array] → 
                        [1, 2, 0, 2] 
    ↓ 
check ranges inside [Left=0, Right=3] → 
                                        shortest bar = ? → area = ?


```python
min_h = float('inf')
for k in range(left, right + 1):
    if heights[k] < min_h:
        min_h = heights[k]
width = right - left + 1
area = min_h * width
if area > best:
    best = area
```    
---

Updated todo list

You already have the histogram idea in place in max_rectangle.ipynb: the `heights` array is the histogram.

The rule is simple:

- Start with `heights = [0] * cols`
- For each row:
  - If the cell is `"1"`, increase that column’s height by 1
  - If the cell is `"0"`, reset that column’s height to 0

That means each position in `heights` becomes the number of consecutive 1s stacked upward ending at the current row.

Example:

```text
matrix rows:
1 0 1 0 0
1 0 1 1 1
1 1 1 1 1
1 0 0 1 0
```

After each row, the histogram heights become:

- After row 1: `[1, 0, 1, 0, 0]`
- After row 2: `[2, 0, 2, 1, 1]`
- After row 3: `[3, 1, 3, 2, 2]`
- After row 4: `[4, 0, 0, 3, 0]`

So the matrix problem turns into a repeated histogram problem: for each row, treat `heights` like bar heights and find the biggest rectangle you can form there.

The key mental model is this: each row is the “floor,” and `heights` tells you how tall each column is if you stack 1s upward from that floor.

If you want, I can next explain how to get the max rectangle from one histogram step by step using your current example.

*   The column number (start_col)
*   The shortest bar value (min_h)


In [7]:
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        # So the update rule is: if current cell is 1, height = previous row's height + 1. If it's 0, height = 0.
        if not matrix: 
            return 0

        rows, cols = len(matrix), len(matrix[0])
        heights = [0] * cols
        best = 0
        for i in range(len(matrix)):
            # matrix[i] is the current row
            
            for j in range(len(matrix[i])):
                # matrix[i][j] is "1" or "0"
                if matrix[i][j] == "1":
                    heights[j] += 1
                else:
                    heights[j] = 0
            
            for start_col in range(cols):
                for end_col in range(start_col, cols):
                    # find smallest height between left and right
                    # count columns: right - left + 1
                    # area = min_height * count
                    # best = max(best, area)
                    min_h = float('inf')
                    for k in range(start_col, end_col + 1):
                        if heights[k] < min_h:
                            min_h = heights[k]
                    width = end_col - start_col + 1
                    area = min_h * width
                    if area > best:
                        best = area
                    
        return best

if __name__ == '__main__':
    sol = Solution()
    # Test your cases here
    print(sol.maximalRectangle([["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]))    # Should print: abc
    print(sol.maximalRectangle([["1","0","1","0","0"],["1","1","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]])) # Should print: acdb

6
10


ok so then what if it is
10100
01011
11111
10101
010101
11111

then what would it? Because somehow after row 4 you have 4, 0, 0, 3, 0, meaning whenever there is a 0 we start over, which makes sense now considering we need to be consecutive, but that just shows consecutive for the first col

Yes, and the important correction is this: the “consecutive” part is tracked independently for every column, not just the first one.

So after each row, heights[j] means:

“How many 1s have I seen in a row in column j, ending at this row?”
That is why a 0 resets only that column. It does not mean the whole rectangle is only about the first column.

Using your first four rows, the height arrays would be:

Row 1: 10100 → [1, 0, 1, 0, 0]
Row 2: 01011 → [0, 1, 0, 1, 1]
Row 3: 11111 → [1, 2, 1, 2, 2]
Row 4: 10101 → [2, 0, 2, 0, 3]
That last one is the key example. It does not mean “only the first column matters.” It means:

column 1 has a vertical streak of 2
column 3 has a streak of 2
column 5 has a streak of 3
Then you check horizontal ranges like:

columns 1 to 1 → height 2, width 1, area 2
columns 3 to 3 → height 2, width 1, area 2
columns 5 to 5 → height 3, width 1, area 3
columns 1 to 3 → minimum height is 0 because column 2 is 0, so that range cannot form a rectangle of 1s
So the histogram step is not “find one column and keep going.” It is “build a bar chart for this row, then ask which adjacent columns can form a solid rectangle.”

One more thing: the matrix you wrote is not rectangular as written. 010101 has 6 characters, while the others have 5. For this problem, all rows need the same length. If you want, I can take your exact example and show the largest rectangle row by row once you confirm whether that fifth row should be 01010 or 010101.